# Dispersion: Where in a Text a Word Appears

**Computational Text Analysis Pack · Module 10 — Visualising Text**

*Technique:* Dispersion plots; distinguishing sustained themes from single episodes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/10-visualising-and-arguing/10c_dispersion_and_structure.ipynb)

Data: `colonial_texts.csv`, `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv', 'masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

Frequency ignores position. A word occurring 60 times might be spread evenly
through a book — a sustained preoccupation — or crammed into one chapter, which is
an episode. Those are different claims and frequency cannot tell them apart.

The **dispersion plot** puts position back: a tick for every occurrence along the
length of the text. It is one of the oldest DH visualisations and one of the few
that is genuinely hard to misread.

## The tooling

**matplotlib**, and the argument that the chart is a claim. **seaborn** and
**ggplot2** (R) are the usual higher-level layers; the decisions this notebook
is about — axis, baseline, ordering — are yours regardless of library.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
# Reassemble one work from its passages, in order.
works = (df_colonial.sort_values("passage_no")
                    .groupby(["work_id", "title", "author"])["text"]
                    .apply(" ".join).reset_index())
works["n_words"] = works["text"].str.split().str.len()
works = works.sort_values("n_words", ascending=False).reset_index(drop=True)
print(works[["title", "author", "n_words"]].head(10).to_string(index=False))

In [ ]:
WORK = 0                    # ← change this
TERMS = ["native", "king", "river", "god", "war", "women"]

row = works.iloc[WORK]
toks = tokenise(row["text"])
print(f"{row['title'][:70]}")
print(f"by {row['author'][:50]}  ({len(toks):,} tokens)")
print()
for term in TERMS:
    print(f"  {term:<12}{toks.count(term):>5} occurrences")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.6))
for i, term in enumerate(TERMS):
    pos = [j for j, w in enumerate(toks) if w == term]
    ax.scatter(pos, [i] * len(pos), marker="|", s=180, linewidths=0.7)
ax.set_yticks(range(len(TERMS))); ax.set_yticklabels(TERMS)
ax.set_xlabel("position in the text (tokens)")
ax.set_xlim(0, len(toks))
ax.set_title(f"Dispersion — {row['title'][:56]}")
plt.tight_layout(); plt.show()

## Read the ticks

- **Evenly spread** — a sustained theme running through the work
- **Clumped in one band** — an episode: one chapter, one incident
- **Front-loaded** — often a term introduced and then abandoned
- **Absent from a long stretch** — a section on a different subject entirely

A clumped word with a high count is the classic false positive of frequency
analysis. It looks like a preoccupation and it is one scene.

In [ ]:
# Put a number on it: how evenly is each word spread?
def dispersion_score(tokens, term, bins=20):
    """Juilland-style evenness: 1 is perfectly even, 0 is all in one place."""
    n = len(tokens)
    edges = np.linspace(0, n, bins + 1).astype(int)
    per_bin = np.array([sum(1 for w in tokens[edges[i]:edges[i + 1]] if w == term)
                        for i in range(bins)], dtype=float)
    if per_bin.sum() == 0:
        return np.nan, per_bin
    mean = per_bin.mean()
    cv = per_bin.std() / mean if mean else np.inf
    return max(0.0, 1 - cv / np.sqrt(bins - 1)), per_bin

print(f"{'term':<12}{'count':>7}{'evenness':>11}   reading")
print("-" * 58)
for term in TERMS:
    score, _ = dispersion_score(toks, term)
    if np.isnan(score):
        continue
    reading = ("sustained" if score > 0.75 else
               "uneven" if score > 0.5 else "episodic")
    print(f"{term:<12}{toks.count(term):>7}{score:>11.2f}   {reading}")

In [ ]:
# Frequency and dispersion together — the two-column table to report.
counts = Counter(toks)
rows = []
for w, n in counts.most_common(400):
    if n < 20 or len(w) < 4:
        continue
    score, _ = dispersion_score(toks, w)
    rows.append({"word": w, "count": n, "evenness": score})
disp = pd.DataFrame(rows)

print("Frequent AND sustained — the themes of the work:")
print(disp[disp["evenness"] > 0.8].head(12)
      .to_string(index=False, float_format=lambda v: f"{v:.2f}"))
print()
print("Frequent but EPISODIC — high counts that are single scenes:")
print(disp[disp["evenness"] < 0.45].head(12)
      .to_string(index=False, float_format=lambda v: f"{v:.2f}"))

### 🔧 Your turn 1

Change `WORK` to a different book and re-run.

Find a word with a high count and a low evenness score, then run KWIC (05a) on
that word within this text. What is the episode?

## Dispersion across a corpus, not within a text

The same idea applies to a corpus with no internal order: instead of position in
a text, use *how many documents* the word appears in. A word occurring 500 times
in 5 documents is a very different object from one occurring 500 times in 400.

In [ ]:
eng = df[df["language"] == "English"].reset_index(drop=True)
tok_docs = [tokenise(t) for t in eng["text"]]
corpus_counts = Counter()
doc_counts = Counter()
for toks in tok_docs:
    corpus_counts.update(toks)
    doc_counts.update(set(toks))

rows = [{"word": w, "occurrences": n, "documents": doc_counts[w],
         "per_doc": n / doc_counts[w]}
        for w, n in corpus_counts.most_common(3000) if n >= 60]
burst = pd.DataFrame(rows).sort_values("per_doc", ascending=False)
print("Words concentrated in few documents — bursty, not general:")
print(burst.head(15).to_string(index=False, float_format=lambda v: f"{v:.1f}"))
print()
print("These are names, places and single running stories. They will dominate a")
print("frequency list and tell you about one event rather than the corpus.")

### 🔧 Your turn 2

Take the top word from that burst list and find which documents it appears in.

Then decide: is it evidence about the corpus, or about one story the corpus
happens to include? Both are legitimate findings — but only if you say which.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Proper names and place names give the most extreme episodic
scores, which is expected: a character or a location belongs to a section of a
narrative. The interesting cases are ordinary nouns with low evenness — a concept
word confined to one chapter usually marks a distinct argumentative or
descriptive passage, and it is a good place to start reading.

**Your turn 2.** Bursty words are almost always a single running story: an
election, an outbreak, a named individual. Reporting them in a general frequency
list without saying so implies a corpus-wide pattern that does not exist. The
one-line fix is to report document frequency next to occurrence count in every
frequency table you publish — it costs a column and prevents the error entirely.

</details>

## Where this stops

You can show where a word lives, not only how often it appears. Next: **Module 11**,
which turns everything in this pack back into an argument.